In [1]:
from PINN_utils import *

In [2]:
pde = PDE(lambda x: 0.6)

In [3]:
N_colocacao = 15000
N_inicial = 1500
N_contorno = 1500
sampler = DomainSampler.create(N_colocacao, N_inicial, N_contorno)

In [4]:
batch_size = 2048
batch_inicial = batch_size // 7
batch_contorno = batch_size // 7
batch_colocacao = batch_size - batch_contorno - batch_inicial

PINN_model = PINN(2, 1, 16, 4)

n_iter = 10000
lr = 1e-2
optimizer = optim.Adam(PINN_model.parameters(), lr=lr)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_iter, eta_min=1e-6)

# a cada quantas iterações vamos alterar os pontos usados
N_change_points = 75

In [5]:
for it in range(n_iter):
    
    batch = sampler.sample_batch(batch_colocacao, batch_inicial, batch_contorno)
    y_fake = PINN_model(batch.get_all())

    L_f = pde.pde_loss(batch.get_colocacao(), y_fake[batch.slice_colocacao])
    L_b1 = pde.IC_loss(batch.get_inicial(), y_fake[batch.slice_inicial])
    L_b2 = pde.BC_loss(batch.get_contorno(), y_fake[batch.slice_contorno])
    
    L_PINN = L_f + 10 * (L_b1 + L_b2)
    
    optimizer.zero_grad()
    L_PINN.backward()
    optimizer.step()
    scheduler.step()

    if it % N_change_points == 0:
        sampler.resample(PINN_model, pde)
    
    if (it + 1) % (n_iter // 10) == 0:
        print(f"Iter {it+1}: L_PINN={L_PINN.item():.10f}")

Iter 1000: L_PINN=0.0169897489
Iter 2000: L_PINN=0.0339227878
Iter 3000: L_PINN=0.0205834173
Iter 4000: L_PINN=0.0162500702
Iter 5000: L_PINN=0.0182925873
Iter 6000: L_PINN=0.0074121868
Iter 7000: L_PINN=0.0079450747
Iter 8000: L_PINN=0.0104365833
Iter 9000: L_PINN=0.0124690831
Iter 10000: L_PINN=0.0120865498


In [6]:
PINN_model.save("resultados/prob3.pth")